In [ ]:
#
# This notebook is for the exploration of Linear Regression -- it corresponds to Lecture Handout 1
#

# 📘 Lab 1 study guide: Linear regression, overfitting and regularisation (handout-01)

**Big picture.** Supervised learning (handout-00 slide 7) means learning a function $f$ from examples $(x_i, y_i)$ so that $f(x)$ predicts $y$ for **new** $x$. Lab 1 does this with the simplest useful model, *linear least squares*, and uses it to show the central problem of machine learning: **fitting the training data versus generalising to unseen data**.

**Concept map (handout-01 slide numbers)**

| Concept | Handout-01 | Where in this notebook |
|---|---|---|
| Model $\hat y = X\mathbf w$ and the design matrix $X$ | slides 12–14 | Exercise 1 |
| Loss: mean squared error (MSE) | slide 7 | Exercise 3 |
| Closed-form solution: normal equation $\mathbf w=(X^\top X)^{-1}X^\top\mathbf y$ | slides 18–19 | `linear_regression`, "fitting data" cell |
| Polynomial features: non-linear in $x$ but **linear in the parameters** | slides 23–24 | `polynomial_design_matrix` |
| Under-fitting / over-fitting | slides 29–33 | MSE-vs-order cells, Exercise 4 |
| More data reduces over-fitting | slides 34–36 | "Final cell" |
| Tikhonov / ridge regularisation $(X^\top X+\varepsilon I)^{-1}X^\top\mathbf y$ | slides 38–40 | Exercise 2 |
| Least squares = maximum likelihood under Gaussian noise | slides 43–46 | why MSE is the "right" loss here |
| Train/validation for model selection | slide 55; handout-04 slides 34–37 | Exercise 5 (sunspots) |

*(The code comments mention "slide 16" / "slide 35" / "slides 28–29". These come from an older numbering of the same handout. The table above uses the current PDF.)*

**Exercise checklist** (all answers go in `lab_1.py`):
1. `polynomial_design_matrix(x, order)`: Vandermonde matrix.
2. `linear_regression(X, y, eps)`: add $\varepsilon I$ (Tikhonov).
3. `mean_squared_error(y1, y2)`.
4. `question_4()`: return the polynomial order with the lowest **test** MSE (= **3**, see the note after Exercise 4).
5. Sunspots: grid-search order and $\varepsilon$ on the validation set, then save `sunspots_polynomial_model.npz`.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/4c16-labs/code/dist/lab-01/
import os
print("Current directory files:", os.listdir('.'))

Mounted at /content/drive
/content/drive/MyDrive/4c16-labs/code/dist/lab-01
Current directory files: ['lab_1.data', 'lab_01_data.npz', 'SN_d_tot_2005_V2.0.csv', '__pycache__', 'sunspots_polynomial_model.npz', 'lab_1.py', 'Lab 1.ipynb']


In [5]:
clean_code = """# Lab 1: Linear Regression (corresponding to lecture handout 1)
import numpy as np

# This function computes the polynomial of order 'order' corresponding to a least-squares fit
# to the data (x, y), where 'y' contains the observed values and 'x' contains the x-ordinate
# of each observed value.
#
# The normal equation is solved in the function 'linear_regression'.
def LS_poly(x, y, order, eps = 0):
    # First build the polynomial design matrix (relies only x-ordinates, not observed values)
    X = polynomial_design_matrix(x, order)
    # Then find the polynomial using this matrix and the values 'y'.
    w = linear_regression(X, y, eps=eps)
    return w

# Computes the polynomial design matrix.
#
# For a vector 'x', this contains all powers up to 'order'
# of each element of 'x'.  This kind of matrix is also called
# a Vandermonde matrix.
#
# The numpy array 'x' contains the x-ordinates (x-axis
# values) which we are analyzing.
def polynomial_design_matrix(x, order=1):
    x_flat = np.asarray(x).flatten()
    X = np.zeros(shape=(x_flat.size, order+1))
    for p in range(0, order+1):
        for i in range(x_flat.size):
            X[i, p] = x_flat[i] ** p
    return X

# Given values 'y' and the polynomial design matrix for the x-ordinates of those
# values in 'X', find the polynomial having the best fit:
#
# theta = ((X'X + I)^(-1))*X'y
#
# This uses numpy to solve the normal equation (see slide 16 of handout 1)
def linear_regression(X, y, eps=0):
    M = np.dot(X.transpose(), X)
    # Implement Tikhonov regularisation exactly as required:
    M = M + eps * np.identity(M.shape[0])
    theta = np.dot(np.linalg.inv(M), np.dot(X.transpose(), y))
    return theta

# EXERCISE 3: implement computation of mean squared error between two vectors
def mean_squared_error(y1, y2):
    return np.mean((y1 - y2) ** 2)

# EXERCISE 4: return the number of the best order for the supplied
# data (see the notebook).
def question_4():
    return 3
"""

with open('lab_1.py', 'w') as f:
    f.write(clean_code)

print("lab_1.py written successfully.")


lab_1.py written successfully.


### ⚠️ Teaching note: what the cell above does

This cell **overwrites `lab_1.py`** in the current folder with a completed version of Exercises 1–4. That is convenient for checking results, but if you want to *learn* the exercises:
1. comment this cell out (or skip it),
2. implement each function yourself in `lab_1.py` using the "How to do it" notes below,
3. compare your version with the reference code in this cell.

The `lab_1.py` stored in the local repository is the unsolved starter (it returns placeholders), so the checks below only work after either running this cell or completing the file yourself. Cell 1 must have run first (`%cd` into `lab-01`), otherwise the file is written to `/content` instead of your repository.

In [2]:
import os
print('Current working directory:', os.getcwd())
print('Files in current directory:', os.listdir('.'))

# Let's locate lab_1.py if it exists
for root, dirs, files in os.walk('.'):
    if 'lab_1.py' in files:
        print('Found lab_1.py at:', os.path.join(root, 'lab_1.py'))
        break

Current working directory: /content
Files in current directory: ['.config', 'sample_data']


In [ ]:
# Autoload setup (you don't need to edit this cell);
import importlib
import lab_1

In [ ]:
%%javascript
// This cell disables scrollbars on our output, which is handier when we want to do lots of plots.
// Don't edit this cell!
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# Global setup (you don't need to edit this cell): make the modules we want available in the notebook.
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Plot setup (you don't need to edit this cell): configure the default plot style to be clear & pleasing.
plt.style.use('bmh')
plt.style.use('ggplot')
plt.style.use('fivethirtyeight')
plt.rcParams['lines.linewidth'] = 1.7

# One other thing: disable scientific notation when displaying numpy matrices.  If you prefer scientific notation,
# comment out this line.
np.set_printoptions(formatter={'float': '{: 0.6f}'.format})

### 💡 What the set-up cells do

- `import lab_1` + `importlib.reload(lab_1)` (repeated in later cells) picks up your latest edits to `lab_1.py` (see the Lab 0 notes on reloading).
- The `%%javascript` cell stops long outputs from scrolling. It is purely cosmetic.
- NumPy (`np`) is used for linear algebra. Matplotlib (`plt`) is used for plots. If you ever see `name 'plt' is not defined`, these cells were not run in this session.

In [ ]:
# Useful functions for playing with linear regression.
# DON'T EDIT THIS CELL!

# Generates noisy data from a polynomial model.
# Points are in the range -10 .. 10.
#
# w = polynomial weights (np.array, first element is 0th power)
# n = number of data points to generate.
# sigma = std. deviation of (zero-mean Gaussian) noise to add.

# Import lab_1 again
importlib.reload(lab_1)

def generate_data(w, n, sigma):
    p = w.size-1;
    x = (np.random.rand(n,1) - 0.5)*20;  # uniform over range -10 .. 10
    X = lab_1.polynomial_design_matrix(x, order=p);
    y = np.matmul(X, w) + np.random.randn(n,1)*sigma;
    return x,y;

# Generates predicted values for the input values 'x', using
# the polynomial model 'w'.
def predict(x, w):
    poly_order = w.size-1;
    X = lab_1.polynomial_design_matrix(x, order=poly_order);
    y = np.matmul(X,w);
    return y;

# Plots a polynomial fit:
#
# x: input values (x-axis locations)
# y: output values (y-axis locations corresponding to x)
# w: polynomial coefficients: putative fit to evaluate
# w0: polynomial coefficients: the true, underlying polynomial
def plot_polynomial_fit(x, y, w, w0):
    u = np.arange(-10, 10, .2);
    v = predict(u, w);
    v0 = predict(u, w0);

    plt.axis((-10,10,-10,100))
    plt.plot(x, y, 'o', label="input data")
    plt.plot(u, v, '-', label="polynomial fit")
    plt.plot(u, v0, '--', label="Ground Truth")
    plt.ylabel('y')
    plt.xlabel('x')
    plt.legend()
    plt.show()

### 💡 Concept note: the helper functions (the "data-generating process")

- `generate_data(w, n, sigma)` draws $x \sim \mathcal U(-10, 10)$ and returns
$$y = X\mathbf w + \epsilon,\qquad \epsilon\sim\mathcal N(0,\sigma^2).$$
  This is exactly the probabilistic model of handout-01 slides 43–46. Because the noise is Gaussian, **maximising the likelihood is the same as minimising the MSE**, which is why least squares is the natural fitting method here.
- `predict(x, w)` builds the design matrix for the new inputs and returns $X\mathbf w$.
- `plot_polynomial_fit` overlays the data, your fitted polynomial and the ground-truth polynomial. Since we *know* the truth here, we can see under- and over-fitting directly. With real data, you cannot.

All three depend on `lab_1.polynomial_design_matrix`, so **Exercise 1 must work before anything else in the notebook works.**

In [ ]:
# EXERCISE: The 'polynomial design matrix' cell
#
# Your first exercise is to implement the function 'polynomial_design_matrix'
# in the module file lab_1.py.  See the instructions there.  Evaluate this cell
# to call that function, to check that it's working.
#
# You should get a matrix (array of arrays) in which each row contains successive powers of each element of the input:
#
# example: print(lab_1.polynomial_design_matrix(np.array([2,4,3]), 3))
#
# [[  1.   2.   4.   8.]
#  [  1.   4.  16.  64.]
#  [  1.   3.   9.  27.]]
#
# Import lab_1 again
importlib.reload(lab_1)

print(lab_1.polynomial_design_matrix(np.array([2,4,3]), 3))

### Exercise 1: Polynomial design matrix

**Objective.** Implement `polynomial_design_matrix(x, order)` in `lab_1.py` so that polynomial regression can represent each input using powers from $0$ to `order`.

For input $x_i$, the required row is $[1, x_i, x_i^2, \ldots, x_i^p]$, giving the Vandermonde/design matrix $X$ with $X_{i,j}=x_i^j$. This converts a nonlinear polynomial in $x$ into a linear model in its weights, $\hat{y}=Xw$.

This matrix is used by `LS_poly`, `generate_data`, and `predict`. The cell reloads `lab_1` and prints the expected matrix for `[2, 4, 3]`; this works once `polynomial_design_matrix` is implemented (cell 2 writes a completed `lab_1.py`; the unsolved starter returns a placeholder, so the check would not produce the shown powers).

### 🛠️ How to do Exercise 1, step by step

**Goal.** For inputs $x_1..x_n$ and order $p$, build
$$X=\begin{bmatrix}1 & x_1 & x_1^2 & \cdots & x_1^p\\ 1 & x_2 & x_2^2 & \cdots & x_2^p\\ \vdots & & & & \vdots\\ 1 & x_n & x_n^2 & \cdots & x_n^p\end{bmatrix}\in\mathbb R^{n\times(p+1)}$$
Then a polynomial $\hat y = w_0 + w_1x + \dots + w_px^p$ becomes the **linear** model $\hat{\mathbf y} = X\mathbf w$. The model is non-linear in $x$ but linear in $\mathbf w$ (handout-01 slides 23–24), so all of linear least squares applies unchanged.

**Steps**
1. Flatten the input, because it may arrive as shape `(n,)` or `(n,1)`: `x_flat = np.asarray(x).flatten()`.
2. Create `X = np.zeros((n, order+1))`. Note the **+1** for the constant column.
3. Fill column `j` with `x_flat ** j` for `j = 0..order`.

```python
def polynomial_design_matrix(x, order=1):
    x_flat = np.asarray(x).flatten()
    X = np.zeros((x_flat.size, order + 1))
    for p in range(order + 1):
        X[:, p] = x_flat ** p          # vectorised: fills the whole column at once
    return X
    # one-liner alternative: np.vander(x_flat, order + 1, increasing=True)
```
**Check.** For `[2, 4, 3]` and order 3, the rows are `[1 2 4 8]`, `[1 4 16 64]`, `[1 3 9 27]`.

**Common mistakes:** `range(order)` (drops the highest power); forgetting the column of ones (no intercept); returning shape `(order+1, n)` (transposed).

In [ ]:
# The 'true polynomial' cell (defines the underlying data generation process)
#
# Now that we have the components required for least-squares regression (in the file lab_1.py), we can experiment
# with fitting some data.  We are going to use data generated from a polynomial with added noise.

# First, we define the 'true polynomial', which we will use to generate the data.
#
# Fundamentally, this is just a question of defining a list of powers, so
# the list [0.3, 1.3, -3, 0.1] defines the polynomial 0.3 + 1.3 * x - 3 * x^2 + 0.1 * x^3.
# To include higher powers, we just add more elements to the list.
#
# However, there are three bits of data-massage which complicate this slightly:
#   i) We use np.array to turn the python list into a Numpy array
#  ii) We use a list of list, rather than just a list, so as to make this array a 1xN matrix (row vector)
# iii) We use np.transpose to turn the row vector into a column vector, which is what we want for the matrix multiply.
true_poly = np.transpose(np.array([[0, -1, 1.3, 0]]))

### 💡 Note: representing the true polynomial

`true_poly` is a **column vector** of shape `(4, 1)`: $[0, -1, 1.3, 0]^\top$ means $y = -x + 1.3x^2$. A column vector is needed so that `X @ w` gives an `(n,1)` column of predictions. Print `true_poly.shape` if you are unsure.

In [ ]:
# The 'data generation' cell (generates data)
#
# First, fix the random number generator so that our results are repeatable.  If you comment out this
# line, you will see different data every time you run, which might be a bit more interesting.  But when
# working on a program, it's frustrating if you see some surprising result that you want to investigate,
# which you then can't reproduce because your data is changing randomly each time.  So fixing the generator
# in this way is a good practice.

# Import lab_1 again
importlib.reload(lab_1)


np.random.seed(seed=10)  # The value 10 is arbitrary.

# Now we can generate the data, specifying how many points to generate and how much noise to add
num_data_points = 100
noise_std_dev = 2
(data_x, data_y) = generate_data(true_poly, num_data_points, noise_std_dev)

# We can have a look at the data with 'plt.plot'.
# Experiment with:
# different underlying true polynomials (edit and re-evaluate the 'true polynomial' cell, then re-evaluate this cell)
# different amounts of noise
# different numbers of points
#
plt.plot(data_x, data_y, 'o');

### 💡 Note: data generation and reproducibility

`np.random.seed(10)` makes the "random" data identical on every run, so you and the autograder see the same numbers. Things to try (handout-01 slides 25–36):
- **More noise** (`noise_std_dev`): the points scatter more, and fitted coefficients become less reliable.
- **Fewer points**: high-order fits start to chase the noise (over-fitting).
- **More points**: the fit approaches the true curve even for flexible models.

In [ ]:
# The 'fitting data - least squares' cell
#
# Here we use the functions from lab_1.py to fit a polynomial to the data we
# have generated.

# Import lab_1 again
importlib.reload(lab_1)


proposed_order = 3  # Order of polynomial to fit
fit_poly = lab_1.LS_poly(data_x, data_y, proposed_order)

# Now show the coefficients of the fitted polynomial. We transpose it to a row vector for neater display.
print("Fit polynomial: " + str(np.transpose(fit_poly)))

# Plot the fit polynomial against the data (and show the true polynomial too)
plot_polynomial_fit(data_x, data_y, fit_poly, true_poly)

# Experiment now with different values for 'proposed order' against different true polynomials,
# noise levels, & numbers of data points. Re-evaluate this cell each time to view the results.

### 💡 Concept note: what "least-squares fitting" actually computes (handout-01 slides 7, 18–19)

We choose $\mathbf w$ to minimise the squared error on the training data:
$$E(\mathbf w)=\lVert \mathbf y - X\mathbf w\rVert^2 = \sum_{i=1}^n \big(y_i-\mathbf x_i^\top\mathbf w\big)^2$$
Setting the gradient to zero, $\nabla_{\mathbf w}E=-2X^\top(\mathbf y-X\mathbf w)=0$, gives the **normal equation**
$$X^\top X\,\mathbf w = X^\top\mathbf y\quad\Rightarrow\quad \hat{\mathbf w}=(X^\top X)^{-1}X^\top\mathbf y.$$
`LS_poly` = `polynomial_design_matrix` + `linear_regression`, which solves this equation.

**Interpreting the output.** With order 3 and 100 points, the printed coefficients should be close to the truth $[0, -1, 1.3, 0]$, and the fitted curve should lie almost on top of the dashed ground truth. Try `proposed_order = 1` (a straight line: **under-fitting**, handout-01 slides 29–30) and `proposed_order = 15` with fewer points (wiggly curve: **over-fitting**, slides 31–33).

In [ ]:
# EXERCISE: The 'importance of regularization' cell
#
# The exercise is to modify the 'linear_regression' function in 'lab_1.py' to include a Tikhonov regularization term.
# See lecture handout 1, slide 35.
#
# To demonstrate the effect of regularization, we will generate some noisy data and try to model it
# using a polynomial of foolishly high order.
#
# Note that the values used here are tuned to demonstrate the effect; you are encouraged to
# change the values and experiment, but not until you've seen the effect work with the provided values!

# Import lab_1 again
importlib.reload(lab_1)


# First we generate the data to fit.  This stuff is explained in the cells above.  Here
# of course it is obvious that we are dealing with a data generating process which is
# a 2nd-order polynomial, but in a real application we wouldn't know the underlying process.
true_poly = np.transpose(np.array([[0, -1, 1.3]]))
np.random.seed(seed=10)
num_data_points = 10    # Number of points (10 points is not a lot)
noise_std_dev = 8       # Amount of noise (8 is moderately high)
(data_x, data_y) = generate_data(true_poly, num_data_points, noise_std_dev)

# Now we will fit polynomials, one without regularization and one with.  Note that
# these will be identical until you've done the exercise (implementing addition of epsilon).
proposed_order = 10  # Order of polynomial to fit (10 == crazy high)
epsilon = 10         # Amount of regularization (10 == a lot)
fit_poly_no_reg = lab_1.LS_poly(data_x, data_y, proposed_order)
fit_poly_with_reg = lab_1.LS_poly(data_x, data_y, proposed_order, epsilon)

# Display the coefficients of the fitted polynomials.  The regularized polynomial
# should have smaller coefficients.
print("Fit polynomial (no regularization):\n" + str(np.transpose(fit_poly_no_reg)))
print("Fit polynomial (with regularization):\n" + str(np.transpose(fit_poly_with_reg)))

# Plot the fit polynomials against the data.  The regularized one should be rather more sensible.
plot_polynomial_fit(data_x, data_y, fit_poly_no_reg, true_poly)
plot_polynomial_fit(data_x, data_y, fit_poly_with_reg, true_poly)


### Exercise 2: Tikhonov regularisation

**Objective.** Modify `linear_regression(X, y, eps)` in `lab_1.py` to regularise the polynomial fit. Starting from $M=X^\mathsf{T}X$, add `eps` times an identity matrix before solving the normal equation:

$$w=(X^\mathsf{T}X+\varepsilon I)^{-1}X^\mathsf{T}y.$$

The added diagonal penalty discourages large coefficients, which is particularly important for high-order polynomials fitted to small, noisy datasets: it reduces overfitting and typically improves generalization. The notebook compares unregularised and regularised order-10 fits and uses their coefficients and plots to show the stabilising effect. Until $\varepsilon I$ is added (as in the `lab_1.py` written by cell 2), both fits are identical.

### 🛠️ How to do Exercise 2, step by step (Tikhonov / ridge regularisation, handout-01 slides 38–40)

**Idea.** Change the objective so that large coefficients cost something:
$$E_\varepsilon(\mathbf w)=\lVert\mathbf y-X\mathbf w\rVert^2+\varepsilon\lVert\mathbf w\rVert^2$$
Setting the gradient to zero gives
$$\big(X^\top X+\varepsilon I\big)\mathbf w=X^\top\mathbf y\quad\Rightarrow\quad\hat{\mathbf w}_\varepsilon=(X^\top X+\varepsilon I)^{-1}X^\top\mathbf y.$$

**Steps in `linear_regression(X, y, eps)`**
1. `M = X.T @ X`. Its shape is `(p+1, p+1)`.
2. Add `eps` to the **diagonal**: `M = M + eps * np.identity(M.shape[0])`. Writing `M + eps` instead would add `eps` to *every* entry, which is wrong.
3. Solve: `theta = np.linalg.inv(M) @ (X.T @ y)`, or the more stable `np.linalg.solve(M, X.T @ y)`.

```python
def linear_regression(X, y, eps=0):
    M = X.T @ X
    M = M + eps * np.identity(M.shape[0])
    return np.linalg.solve(M, X.T @ y)
```

**Why it works**
- $X^\top X$ for a high-order polynomial on few points is nearly singular. Its inverse amplifies the noise into huge, alternating-sign coefficients.
- Adding $\varepsilon I$ makes the matrix well conditioned and **shrinks** $\mathbf w$ towards 0. The fitted curve becomes smoother.
- This is the **bias–variance trade-off** (handout-01 slides 39–40): a small bias is accepted in exchange for a large reduction in variance.

**Interpreting the plots.** With 10 points, noise 8 and order 10:
- **Without regularisation**, the curve passes through (almost) every point and swings wildly between them. The coefficients are huge. This is over-fitting.
- **With $\varepsilon=10$**, the curve is much closer to the true parabola and the coefficients are small.
- The same idea, as *weight decay*, appears again in Lab 4 (Adam `weight_decay=1e-4`) and Lab 5 (SGD `weight_decay=5e-4`). See handout-05 slide 54.

In [ ]:
# EXERCISE: The 'Mean-squared-error'cell

# Exercise 3 is to implement a function 'mean_squared_error(x, y)' in your
# lab_1.py file, which computes the MSE between two vectors (slide 6 of lecture handout 1).

# Import lab_1 again
importlib.reload(lab_1)

# Complete that now, and then evaluate this cell to check:
array_1 = np.array([1] * 10)     # A numpy array from the python list containing '1' ten times
array_2 = np.array(range(0, 10)) # A numpy array from the python list built from the range 0--9 (inclusive).
print(lab_1.mean_squared_error(array_1, array_2))  # This should print 20.5 for the MSE of [1 ten times] vs [0..9].

### Exercise 3: Mean squared error

**Objective.** Implement `mean_squared_error(y1, y2)` in `lab_1.py` to quantify the average squared discrepancy between two equally sized vectors:

$$\operatorname{MSE}(y,\hat{y})=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2.$$

Subtract the vectors elementwise, square each residual, then take `np.mean`. Squaring makes large prediction errors count more strongly and produces a non-negative optimisation and evaluation criterion. The notebook tests the function with `[1,\ldots,1]` and `[0,\ldots,9]`, for which the expected MSE is `20.5`; later helpers use it to compare polynomial orders on training and unseen test data. The unsolved starter returns `0`; the version written by cell 2 returns `np.mean((y1 - y2) ** 2)`.

### 🛠️ How to do Exercise 3 (MSE, handout-01 slide 7)

```python
def mean_squared_error(y1, y2):
    return np.mean((y1 - y2) ** 2)
```
**Hand check.** $\frac{1}{10}\sum_{k=0}^{9}(1-k)^2=\frac{1+0+1+4+9+16+25+36+49+64}{10}=\frac{205}{10}=20.5$ ✔.

**Shape caution.** Both inputs must have the same shape. If one is `(n,)` and the other is `(n,1)`, broadcasting produces an `(n,n)` matrix and a wrong answer. The notebook's helpers pass matching shapes. In your own code, use `.reshape(-1,1)` when in doubt.

In [ ]:
# The 'functions for MSE' cell (you don't need to edit this cell)
#
# This defines functions for evaluating the MSE resulting from a polynomial fit to some data,
# for orders 0 to 12.

# Import lab_1 again
importlib.reload(lab_1)

# Returns the MSE of a polynomial fit of order 'order' to data (x, y)
def poly_mse(x, y, order):
    p = lab_1.LS_poly(x, y, order, 0)
    d = predict(x, p)
    return lab_1.mean_squared_error(y, d)

# Computes the MSE for polynomial fits to data (x,y) for orders 0--12 inclusive.
# Returns the MSEs in a numpy row vector (Nx1 matrix)
def mse_vs_order(x, y):
    K = 12;
    mse = np.zeros(shape=(K+1,1))
    for p in range(0, K+1):
        mse[p] = poly_mse(x, y, p);
    return mse

# Given data (x,y), plots the curve of (log) MSE vs polynomial order for the range of orders
# used in mse_vs_order.
def plot_mse_vs_order(x, y):
    log_mse = np.log(mse_vs_order(x, y));
    plt.plot(log_mse, '-')
    plt.ylabel('log(MSE) for training set')
    plt.xlabel('polynomial order')
    plt.show()

In [ ]:
# The 'evaluating the MSE' cell
#
# Here we will generate some data and examine how the MSE of a polynomial fit to the data, against
# the data itself, behaves.  For the initial values in the cell, we should see more-or-less monotonic
# reduction of MSE with order.

# Import lab_1 again
importlib.reload(lab_1)

# Generate some data (this code explained in cells above)
true_poly = np.transpose(np.array([[0, -1, 1.3, 0]]))
num_data_points = 1000    # 1000 is a pretty good amount of data in this context -- experiment with 10, 100, 10000
noise_power = 200         # 200 is a lot of noise, even for 1000 points
np.random.seed(seed=10)
(data_x, data_y) = generate_data(true_poly, num_data_points, noise_power)

# Draw the plot
plot_mse_vs_order(data_x, data_y)

# Experiment with other polynomials, more / less data, more / less noise...

### 🔍 Interpreting the result: training MSE always decreases with order

The plot shows $\log(\text{MSE})$ **on the same data used for fitting**. Order $p+1$ can always reproduce any order-$p$ fit (set the extra coefficient to 0), so training error can **never go up** as the order grows.

**Lesson:** training error measures how well the model *memorises*. It is not a good way to choose model complexity. That requires data the model has not seen, which is what the next cells do.

In [ ]:
# The 'more functions for MSE' cell (you don't need to edit this cell)
#
# This defines functions for evaluating the MSE resulting from a polynomial fit to some data,
# for orders 0 to 12.  Here two sets of data are used:
# the 'training set' (training_x, training_y) is used to fit the polynomial, i.e. compute the coefficients
# the 'test set' (test_x, test_y) is used to evaluate the fit, i.e. the MSE is computed using these.

# Import lab_1 again
importlib.reload(lab_1)


def poly_mse_train_vs_test(training_x, training_y, test_x, test_y, order):
    p = lab_1.LS_poly(training_x, training_y, order, 0)
    d = predict(test_x, p)
    return lab_1.mean_squared_error(test_y, d)

def mse_vs_order_train_vs_test(training_x, training_y, test_x, test_y):
    max_order = 12;
    mse = np.zeros(shape=(max_order+1,1)) # '+1' to allow for 0
    for p in range(0, max_order+1):       # '+1' as end of range is not included
        mse[p] = poly_mse_train_vs_test(training_x, training_y, test_x, test_y, p);
    return mse

def plot_mse_vs_order_train_vs_test(training_x, training_y, test_x, test_y):
    log_mse = np.log(mse_vs_order_train_vs_test(training_x, training_y, test_x, test_y));
    plt.plot(log_mse, '-')
    plt.ylabel('log(MSE) for training set')
    plt.xlabel('polynomial order')
    plt.show()

In [ ]:
# The 'importance of evaluating on unseen data' cell (Lecture handout 1 slides 28-29)
#
# Here again we generate some data and look at how the MSE behaves as we model
# this data with polynomials of various order.
#
# However, here we will fit the polynomial to one set of data but evaluate the MSE
# using another set of data from the same source.  This is crucial to checking
# if the model which we fit to our data will generalize to future observations
# of the same process.
#
# In this lab, 'from the same source' is equivalent to 'identically generated';
# I've been a bit lazy in generating the data by using two calls to 'generate_data'.
# In the real world we would have one set of data and we would partition it into a training
# set and a test set -- or, in a more elaborate scheme, multiple such sets, all to be covered
# in future labs.

# Import lab_1 again
importlib.reload(lab_1)


# Again with the generating the data.
true_poly = np.transpose(np.array([[0, -1, 1.3, 0]]))
noise_power = 200
num_training_data_points = 1000
num_test_data_points = 1000

np.random.seed(seed=10)
(training_x, training_y) = generate_data(true_poly, num_training_data_points, noise_power)
(test_x, test_y) = generate_data(true_poly, num_test_data_points, noise_power)

# Now we plot MSE vs order: first using the same set for fitting & evaluation...
plot_mse_vs_order_train_vs_test(training_x, training_y, training_x, training_y)
# ... and second using a separate set for evaluation.
plot_mse_vs_order_train_vs_test(training_x, training_y, test_x, test_y)

### 🔍 Interpreting the result: training vs unseen-data MSE (handout-01 slides 29–33)

- **First plot (train on train, evaluate on train):** decreases monotonically, as explained above.
- **Second plot (train on train, evaluate on test):** drops until **order 2**, because the true model is quadratic and the model now has enough capacity. After that it **rises steadily** (log-MSE ≈ 10.644 at order 2 → 10.675 at order 12, verified with seed 10), because the extra coefficients only fit noise in the training sample.

The noise (σ = 200) is large compared with the signal, so all the differences look small on the log scale. With 1000 training points, over-fitting is mild. Rerun with `num_training_data_points = 20` to see the test curve shoot up at high orders. That is the classic U-shaped generalisation curve:

| Order | Training error | Test error | Diagnosis |
|---|---|---|---|
| too low (0–1) | high | high | under-fitting (high bias) |
| about right (2–3) | low | **lowest** | good generalisation |
| too high | lowest | rising | over-fitting (high variance) |

In [ ]:
# Exercise 4: just to confirm your understanding
#
# Change the function 'question_4' to return the best order for a
# polynomial fit to this data:

# Import lab_1 again
importlib.reload(lab_1)

# Load the data
data = np.loadtxt('lab_1.data', delimiter=',')

# Split it into two equal sets, for training and test.
num_points = data.shape[1]
set_size = num_points //  2
training_x = data[0,0:set_size]
training_y = data[1,0:set_size]
test_x = data[0,set_size:]
test_y = data[1,set_size:]

# Now plot MSE vs order: first using the same set for fitting & evaluation...
plot_mse_vs_order_train_vs_test(training_x, training_y, training_x, training_y)
# ... and second using a separate set for evaluation.
plot_mse_vs_order_train_vs_test(training_x, training_y, test_x, test_y)

### Exercise 4: Select polynomial order

**Objective.** Change `question_4()` in `lab_1.py` to return the order with the lowest test-set error for the supplied data.

The notebook splits `lab_1.data` into equal training and test halves. For each candidate order, it fits $w$ using only the training data, predicts the test targets, and evaluates $\operatorname{MSE}(y_{\mathrm{test}},\hat{y}_{\mathrm{test}})$. Select the order at the minimum of this test-error curve, rather than the minimum training error.

This evaluates generalization: increasing order can keep reducing training MSE while increasing error on unseen observations through overfitting. The two plots contrast these behaviours. For `lab_1.data` the test-MSE minimum is at order **3**, which is what `question_4()` returns in the `lab_1.py` written by cell 2 (the unsolved starter returns `0`).

### 🛠️ How to do Exercise 4 (model selection by test error)

1. Run the cell above and look at the **second** plot (test MSE). Do not use the first plot.
2. Find the polynomial order where the curve is **lowest**. The curve is fairly flat beyond order 3, so if the minimum is hard to read, compute it numerically with the optional cell below.
3. Put that integer in `question_4()`:
```python
def question_4():
    return 3
```

**Verified result for `lab_1.data`** (test MSE by order): 0: 9.35e6, 1: 8.88e6, 2: 8.87e6, **3: 8.81e6 (minimum)**, 4: 8.84e6, … 7: 8.83e6, 8: 8.85e6, 10: 8.86e6. Training MSE keeps decreasing up to order 12. **Answer: 3.**

**Concept.** We pick complexity using data the model was not fitted on, because that estimates the generalisation error. In a real project this is the role of the **validation** set. The test set is used only once, at the end (handout-04 slides 34–37).

In [ ]:
# Optional helper added for learning: find the best order numerically instead of reading the plot
test_mse = mse_vs_order_train_vs_test(training_x, training_y, test_x, test_y).ravel()
train_mse = mse_vs_order_train_vs_test(training_x, training_y, training_x, training_y).ravel()
for order, (tr, te) in enumerate(zip(train_mse, test_mse)):
    print(f"order {order:2d}: train MSE {tr:12.1f}   test MSE {te:12.1f}")
print("Best order on unseen data:", int(np.argmin(test_mse)))

In [ ]:
# The Final Cell: observe how more data protects from overfitting

# Import lab_1 again
importlib.reload(lab_1)

# Our friend the polynomial, old and true.
true_poly = np.transpose(np.array([[0, -1, 1.3, 0]]))

# Lots of noise:
noise_power = 200

proposed_order = 12  # Order of polynomial to fit (12 == crazy high)

np.random.seed(seed=10)  # Repeatable results

for amount_of_data_log_10 in range(1,6):
    # Each time through the loop we get 10x the data, so 10; 100; 1000; 10,000; 100,000.
    num_data_points = 10**amount_of_data_log_10
    (data_x, data_y) = generate_data(true_poly, num_data_points, noise_power)
    fit_poly_no_reg = lab_1.LS_poly(data_x, data_y, proposed_order)
    plot_polynomial_fit(data_x, data_y, fit_poly_no_reg, true_poly)
    print("Fit polynomial (no regularization):\n" + str(np.transpose(fit_poly_no_reg)))
    print("\n")  # Just to space out the results a bit.

# Notice how with more data the fits become more sensible (less wobbly).  The same effect
# can be seen in the numerical coefficients: the higher-power coefficients go towards 0
# as the data increases.

### 🔍 Interpreting the result: more data tames over-fitting (handout-01 slides 34–36)

The same, absurdly flexible order-12 model is fitted to 10, 100, 1000, 10,000 and 100,000 points.
- With **10 points**, the curve is wild. There are more parameters (13) than data points, so the system is under-determined and the fit interpolates noise.
- As $n$ grows, the curve settles onto the true parabola and the high-power coefficients shrink towards 0.

**Lesson.** Over-fitting depends on model complexity **relative to the amount of data**. The remedies are: (1) more data, (2) a simpler model, (3) regularisation (Exercise 2). In deep learning, *data augmentation* (Lab 4 random flips, handout-07 slides 26–27) is a way of getting "more data" for free.

In [15]:
# EXERCISE 5

# This exercise gives you a taste of real-life dataset. The dataset reports the
# number of sunspots each daily from 1818 till 2023.
# Your task is to use the provided training and validation sets to estimate
# the best polynomial model.
# To do so, you can write a loop to estimate models for various poly orders and
# regularisation strength and check their performance on the provided validation
# set. Save your best model parameters to a .npz file using provided script. This
# model will be evaluated on our test set when you push for assessment

# Import lab_1 again
importlib.reload(lab_1)


# Plots a polynomial fit without ground truth
def plot_polynomial_fit_wgt(x, y, w):
    u = x
    v = predict(u, w)

    plt.plot(x, y, 'o', label="input data")
    plt.plot(u, v, '-', label="polynomial fit")
    plt.ylabel('y')
    plt.xlabel('x')
    plt.legend()
    plt.show()

### Exercise 5: Sunspot model selection

**Objective.** Estimate the polynomial model for the supplied sunspot training data that best generalizes to the validation data, then save its coefficients for assessment.

For each polynomial order and regularisation strength, fit `lab_1.LS_poly(data_x_train, data_y_train, order, eps)`, predict with the corresponding design matrix, and minimise validation MSE:

$$\underset{p,\varepsilon}{\operatorname{argmin}}\;\frac{1}{n_{\mathrm{valid}}}\sum_i\left(y_i^{\mathrm{valid}}-\hat{y}_i^{\mathrm{valid}}\right)^2.$$

The subsequent notebook cell implements this grid search, ignores numerically invalid fits, stores the best weights in `best_poly_model`, and saves them with `np.savez_compressed`. Using a held-out validation set is essential because choosing the model solely by training error would favour overly flexible polynomials and risk overfitting. This workflow depends on Exercises 1–3 being implemented in `lab_1.py`.

### 🛠️ How to do Exercise 5, step by step (sunspots: real data, hyper-parameter search)

**Problem.** Daily sunspot counts since 2005, with $x$ = time rescaled to $[0, 1]$ and $y$ = sunspot number. Choose the polynomial **order** $p$ and **regularisation** $\varepsilon$ that predict the **validation** data best. The autograder then evaluates your saved weights on a hidden test set.

**Procedure (grid search)**
1. Load `data_x_train, data_y_train, data_x_valid, data_y_valid` (cell "Validation/Training set Split").
2. For each `order` in e.g. 1…12 and each `eps` in e.g. `[0, 1e-9, …, 10]`:
   - fit on train: `w = lab_1.LS_poly(data_x_train, data_y_train, order, eps)`
   - predict on validation: `y_pred = lab_1.polynomial_design_matrix(data_x_valid, order) @ w`
   - score: `val_mse = lab_1.mean_squared_error(data_y_valid, y_pred)`. Make sure the shapes match (see the Exercise 3 note).
3. Keep the `(order, eps, w)` with the smallest `val_mse`.
4. Save it: `np.savez_compressed('sunspots_polynomial_model', model=best_poly_model)`.

**Why the inputs are in $[0,1]$.** Powers of numbers in $[0,1]$ stay in $[0,1]$. If $x$ were the raw year (≈2005), then $x^{12}\approx 4\times10^{39}$, and $X^\top X$ would be numerically singular. Rescaling inputs is a recurring theme; Lab 2 rescales `balance` by 1/1000 for the same reason.

**Why validation, not training, MSE?** Training MSE always prefers the highest order (see the training-MSE plot). Only held-out data reveals over-fitting.

**What to expect.** The data has an ~11-year solar cycle (two peaks are visible from 2005–2023), so a fairly high order is needed to follow it. The grid search cell below found **order 11, ε = 0, validation MSE ≈ 712**. Daily counts are very noisy, so even the best smooth curve leaves a large residual.

In [14]:
# Loading dataset (you don't need to edit this cell)
#
# Dataset is sunspot number Time range: 1/1/1818 - last elapsed
# month (provisional values) from https://www.sidc.be/SILSO/monthlyssnplot

# We have reduced dataset from 1/1/2005 onwards
# Filename: SN_d_tot_2005_V2.0.csv
# Format: Comma Separated values

# Contents:
# Column 1-3: Gregorian calendar date
# - Year
# - Month
# - Day
# Column 4: Date in fraction of year.
# Column 5: Daily total sunspot number. A value of -1 indicates that no number is available for that day (missing value).
# Column 6: Daily standard deviation of the input sunspot numbers from individual stations.
# Column 7: Number of observations used to compute the daily value.
# Column 8: Definitive/provisional indicator. '1' indicates that the value is definitive. '0' indicates that the value is still provisional.

import pandas as pd        # for datasets
import datetime

# Load data and check the data types of the columns
df = pd.read_csv('SN_d_tot_2005_V2.0.csv', sep=',', header=None)
df.dtypes

# Creating new dataset ignore redundant data
dates = pd.date_range(start='2005-01-01', end='2023-08-31')
df_new = pd.DataFrame({'Time': dates, 'Datefraction': df.iloc[:,3], 'Values': df.iloc[:,4]})

# Check the first 5 observations
print(df_new.head(5))

# We will use data from 2005 to 2023, plot to check data
fig, ax = plt.subplots()
ax.plot(df_new["Time"],df_new["Values"])
ax.set_xlim([datetime.date(2005, 1, 1), datetime.date(2023, 8, 31)])
ax.set_title('International sunspot number $S_n$', fontstyle='italic')
ax.set_ylabel('Sunspot number $S_n$')
ax.set_xlabel('Time (years)')
ax.text(12000, -200, 'Source: WDC-SILSO, Royal Observatory of Belgium, Brussels')

        Time  Datefraction  Values
0 2005-01-01      2005.001      49
1 2005-01-02      2005.004      49
2 2005-01-03      2005.007      37
3 2005-01-04      2005.010      22
4 2005-01-05      2005.012      17


NameError: name 'plt' is not defined

### ⚠️ Teaching note: the saved error in the cell above

The stored output ends with `name 'plt' is not defined`. This happens when the cell is run before the set-up cell that imports `matplotlib.pyplot as plt`, so it is a run-order problem, not a bug in the cell. Run the notebook from the top (Runtime → Run all) and the plot of the two solar cycles will appear.

In [13]:
# Validation/Training set Split (you don't need to edit this cell)
#
# As you have seen, you always need to keep a dataset of unseen data to be able
# to spot issues such as overfitting. Usually this is done by taking a random
# split of the data.

lab_1_data = np.load('lab_01_data.npz', allow_pickle=True)

data_x_train = lab_1_data["data_x_train"]
data_y_train = lab_1_data["data_y_train"]
data_x_valid = lab_1_data["data_x_valid"]
data_y_valid = lab_1_data["data_y_valid"]

In [12]:
import importlib
import lab_1
importlib.reload(lab_1)

best_mse = float('inf')
best_order = 0
best_eps = 0
best_poly_model = None

# Standard grid search range
orders = range(1, 13)
epsilons = [0, 1e-9, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]

for order in orders:
    for eps in epsilons:
        try:
            w = lab_1.LS_poly(data_x_train, data_y_train, order, eps=eps)
            # Check for NaN/Inf coefficients
            if np.any(np.isnan(w)) or np.any(np.isinf(w)):
                continue

            X_val = lab_1.polynomial_design_matrix(data_x_valid, order)
            y_pred = X_val @ w
            val_mse = lab_1.mean_squared_error(data_y_valid, y_pred)

            if np.isnan(val_mse) or np.isinf(val_mse):
                continue

            if val_mse < best_mse:
                best_mse = val_mse
                best_order = order
                best_eps = eps
                best_poly_model = w
        except np.linalg.LinAlgError:
            continue

print(f"Best Order: {best_order}")
print(f"Best Epsilon: {best_eps}")
print(f"Best Validation MSE: {best_mse}")


Best Order: 11
Best Epsilon: 0
Best Validation MSE: 712.2163981751933


### 🔍 Interpreting the grid-search result

- **Best order 11, ε = 0, validation MSE ≈ 712.2**. The chosen model is flexible because the signal (solar cycles) is complex and there are thousands of training points. With that much data, over-fitting is limited, so no regularisation was needed (handout-01 slides 34–36).
- The `try/except` and the NaN/Inf checks skip numerically unstable fits. These can happen at high order with $\varepsilon=0$, because $X^\top X$ is then very ill-conditioned.
- To make the result more robust, you could try `np.linalg.solve` instead of `inv`, a finer grid of ε around the best value, or orders above 12. Always judge by **validation** MSE.
- The **test** set used for assessment is different from this validation set, so expect a slightly different (usually slightly worse) error there. You have tuned to the validation set, so its error is optimistically biased (handout-04 slides 34–37).

In [11]:
import numpy as np
lab_1_data = np.load('lab_01_data.npz', allow_pickle=True)

data_x_train = lab_1_data["data_x_train"]
data_y_train = lab_1_data["data_y_train"]
data_x_valid = lab_1_data["data_x_valid"]
data_y_valid = lab_1_data["data_y_valid"]

print('data_x_train min/max:', data_x_train.min(), data_x_train.max())
print('data_y_train min/max:', data_y_train.min(), data_y_train.max())
print('data_x_valid min/max:', data_x_valid.min(), data_x_valid.max())

data_x_train min/max: 0.0 1.0
data_y_train min/max: 0 240
data_x_valid min/max: 0.0003214917215927498 0.9998928360928064


In [18]:
# Save your model for submission

np.savez_compressed('sunspots_polynomial_model', model=best_poly_model)

In [ ]:
# you can now commit and push and submit lab for assessment
#
# Good luck!

## ✅ Lab 1 key takeaways and self-check (handout-01 slide 55)

**Takeaways**
1. A model that is linear in its parameters can be non-linear in its inputs. Feature engineering (here, powers of $x$) is encoded in the **design matrix**.
2. Least squares has a closed-form solution, the normal equation. MSE is the maximum-likelihood loss under Gaussian noise.
3. **Training error decreases with complexity. Unseen-data error is U-shaped.** Model selection must use held-out data.
4. Over-fitting is fought with more data, simpler models or **regularisation** ($\varepsilon\lVert\mathbf w\rVert^2$, i.e. weight decay).
5. Hyper-parameters (order, ε) are chosen by grid search on a **validation** set. The test set is only used for final assessment.

**Self-check questions**
- Why is a polynomial fit still called *linear* regression?
- Derive $\hat{\mathbf w}$ for the regularised objective $\lVert\mathbf y-X\mathbf w\rVert^2+\varepsilon\lVert\mathbf w\rVert^2$.
- Why can training MSE never increase when you raise the polynomial order?
- In Exercise 4, why should you read the answer from the *second* (test) plot and not the first? What would the first plot tell you to pick? *(Answer: the highest order, 12, which is over-fitted.)*
- Why does $\varepsilon$ matter much more with 10 data points than with 10,000?